[Attention is all you need!](https://arxiv.org/pdf/1706.03762)

[Transformers, the tech behind LLMs | Deep Learning Chapter 5](https://www.youtube.com/watch?v=wjZofJX0v4M)

## Modelowanie Języka z Wykorzystaniem Sieci Transformatorowych

Laboratoria skupiają się na przełożeniu teorii na działający kod, budując od zera architekturę zaproponowaną w pracy *"Attention Is All You Need"*. Naszym celem jest zrozumienie mechaniki działania modeli pozbawionych warstw rekurencyjnych i konwolucyjnych, bazujących wyłącznie na mechanizmach uwagi.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/20251004125140134507/transformers.webp" width=70%><br><i>www.geeksforgeeks.org</i></center>

### Etapy Realizacji Projektu

Pracę nad modelem zrealizujemy w trzech głównych fazach, łącząc poszczególne operacje tensorowe w coraz większe moduły:



#### **Faza I: Reprezentacja i Pozycjonowanie**

W pierwszym kroku musimy przetworzyć dyskretne tokeny wejściowe (w postaci macierzy o wymiarach `[wielkość_partii, długość_sekwencji]`) na ciągłą przestrzeń wektorową. Zaimplementujemy warstwę rzutowań (*Embeddings*), która rozszerzy nasz tensor o dodatkowy wymiar modelu (w oryginalnej pracy przyjęto $d_{model}=512$ ). Ponieważ nasza sieć nie przetwarza danych sekwencyjnie, w tym miejscu konieczne będzie również dodanie sygnału o pozycji każdego słowa za pomocą funkcji trygonometrycznych (kodowanie pozycyjne).




#### **Faza II: Inżynieria Mechanizmu Uwagi**

Sercem naszego modelu będzie moduł *Scaled Dot-Product Attention*. Przy jego implementacji kluczowe będzie zwrócenie uwagi na prawidłowe skalowanie wyników przez odwrotność pierwiastka z wymiaru klucza ($\frac{1}{\sqrt{d_k}}$)  – pominięcie tego kroku drastycznie pogarsza stabilność uczenia.

Następnie obudujemy ten mechanizm w architekturę wielogłową (*Multi-Head Attention*), co wymusi na nas operowanie na tensorach o kształcie `[wielkość_partii, liczba_głów, długość_sekwencji, długość_sekwencji]`. W tej fazie przygotujemy również logikę maskowania wartości. Maska przyczynowa w dekoderze będzie kluczowa, aby zablokować modelowi dostęp do przyszłych kontekstów, co zrealizujemy poprzez ustawienie odpowiednich wartości na $-\infty$ tuż przed aplikacją funkcji softmax.

#### **Faza III: Składanie Warstw Ukrytych i Kompozycja Modelu**

Gdy moduły uwagi będą gotowe, połączymy je z w pełni połączonymi sieciami jednokierunkowymi (*Feed-Forward Networks*). Warto pamiętać, że wewnętrzna warstwa ukryta tych sieci znacząco rozszerza wymiarowość (standardowo do $d_{ff}=2048$ ).

Każda podwarstwa zostanie otoczona mechanizmem normalizacji (*Layer Normalization*) oraz połączeniem omijającym (rezydualnym). Na tym etapie należy bezwzględnie pilnować zgodności kształtów tensorów na wejściu i wyjściu poszczególnych bloków. Gotowe bloki ułożymy w stosy kodera i dekodera (po 6 takich warstw ), a proces zakończymy transformacją liniową i nałożeniem funkcji softmax w celu wygenerowania ostatecznych prawdopodobieństw dla słownika.


### Weryfikacja i Optymalizacja Złożoności

Podczas pisania kodu zachęcam do testowania każdej funkcji z osobna, upewniając się, że gradienty przepływają swobodnie, a maski faktycznie zerują niepożądane połączenia. Należy mieć na uwadze, że samo-uwaga charakteryzuje się kwadratową złożonością pamięciową i obliczeniową względem długości analizowanego tekstu ($O(n^2 \cdot d)$). Będzie to wymagało od Was rozważnego dobierania rozmiaru paczki danych (batch size) podczas późniejszych prób treningowych, aby nie przekroczyć dostępnej pamięci VRAM. Nie zapomnijcie również o zaimplementowaniu mechanizmu odrzucania (*Dropout*)  jako formy regularyzacji.

### Zadanie 1: Zamiana słów na wektory (Embeddings) i Informacja o Pozycji

W tym zadaniu zbudujecie moduł wejściowy sieci. Transformery różnią się od starszych sieci (rekurencyjnych) tym, że nie przetwarzają tekstu słowo po słowie, tylko widzą całe zdanie od razu. Z tego powodu musimy przekazać modelowi nie tylko to, co oznaczają poszczególne słowa, ale też na którym miejscu w zdaniu się one znajdują.

**Wskazówki do napisania kodu:**

* **Zamiana tokenów na wektory (Embeddings) -** Użyjcie gotowego modułu `nn.Embedding` z biblioteki PyTorch, żeby zamienić słowa na gęste wektory. Załóżcie, że początkowy rozmiar wektora słowa może być czasem inny niż główny rozmiar modelu (wtedy trzeba użyć dodatkowej warstwy liniowej, by je zrównać). Na samym początku funkcji `forward` dodajcie zabezpieczenie (asercję), które sprawdzi, czy dane wejściowe mają poprawny typ – w tym przypadku powinien to być `torch.long`.

* **Skalowanie wektorów -** Ważnym krokiem opisanym w artykule jest pomnożenie wektorów słów przez pierwiastek kwadratowy z głównego rozmiaru modelu (czyli $\sqrt{d_{model}}$). Żeby kod działał szybciej, policzcie ten pierwiastek tylko raz i zapiszcie go jako ułamek (`float`) już w konstruktorze `__init__`.

* **Dodanie informacji o pozycji słowa -** Żeby model wiedział, w jakiej kolejności ułożone są słowa w zdaniu, stwórzcie kodowanie pozycji oparte na funkcjach trygonometrycznych – sinusach i kosinusach. Ponieważ informacje o pozycji oraz wektory słów muszą mieć ten sam rozmiar, po prostu dodajemy je do siebie. Pamiętajcie tylko o dopasowaniu kształtu tensora pozycji – trzeba go "rozszerzyć" o wymiar paczki danych (*batch dimension*), żeby dodawanie zadziałało bez błędów.

* **Zabezpieczenie przed przeuczeniem (Dropout) -** Na samym końcu, po dodaniu do siebie wektorów słów i wartości pozycji, przepuśćcie ten wynik przez warstwę *Dropout*.


In [ ]:
import torch
import torch.nn as nn
import math

class PositionalSignal(nn.Module):
    """
    Moduł dodający kodowanie pozycyjne do embeddingów.
    Tworzy macierz pozycji raz w __init__, a potem tylko ją przycina w forward().
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # TODO (student)
        # 1) Utwórz tensor na kodowanie pozycyjne o kształcie [max_len, d_model]
        #    Użyj: torch.zeros(...)
        pe =

        # 2) Utwórz indeksy pozycji [0, 1, 2, ... max_len-1]
        #    Kształt ma być [max_len, 1], dlatego dodaj wymiar przez .unsqueeze(1)
        #    Użyj: torch.arange(..., dtype=torch.float).unsqueeze(1)
        position =

        # 3) Oblicz współczynniki skalujące dla parzystych wymiarów embeddingu
        #    (częstotliwości dla sin/cos)
        #    Użyj: torch.arange(0, d_model, 2), torch.exp(...), math.log(...)
        #    Wynik powinien mieć kształt [d_model // 2]
        div_term =

        # 4) Wpisz sinusy do parzystych kolumn (0, 2, 4, ...)
        #    Użyj indeksowania pe[:, 0::2] oraz torch.sin(...)
        pe[:, 0::2] =

        # 5) Wpisz cosinusy do nieparzystych kolumn (1, 3, 5, ...)
        #    Użyj indeksowania pe[:, 1::2] oraz torch.cos(...)
        pe[:, 1::2] =

        # 6) Dodaj wymiar batcha z przodu, aby uzyskać kształt [1, max_len, d_model]
        #    Użyj: pe.unsqueeze(0)
        #
        # 7) Zarejestruj tensor jako buffer:
        #    - nie jest trenowany (to nie są wagi),
        #    - ale zapisuje się razem z modelem,
        #    - i przenosi się poprawnie na CPU/GPU.
        #    Użyj: self.register_buffer("nazwa", tensor)
        self.register_...

    def forward(self, x):
        # x ma zwykle kształt: [batch_size, seq_len, d_model]
        # Pobieramy tylko tyle pozycji, ile wynosi długość aktualnej sekwencji (seq_len)
        # x.size(1) = seq_len
        #
        # self.pe_matrix[:, :x.size(1)] ma kształt [1, seq_len, d_model]
        # PyTorch zastosuje broadcasting po wymiarze batch_size przy dodawaniu.
        return x + self.pe_matrix[:, :x.size(1)]


class TransformerInput(nn.Module):
    """
    Moduł wejściowy transformera:
    tokeny -> embeddingi -> (opcjonalna projekcja) -> dodanie pozycji -> dropout
    """
    def __init__(self, vocab_size, d_model, d_embed, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model

        # TODO (student)
        # 1) Warstwa embedding:
        #    zamienia indeksy tokenów (int) na wektory o rozmiarze d_embed
        #    Użyj: nn.Embedding(vocab_size, d_embed)
        self.word_lookup =

        # 2) Jeśli d_embed != d_model, trzeba dopasować wymiar embeddingu do wymiaru modelu.
        #    - gdy różne: użyj nn.Linear(d_embed, d_model)
        #    - gdy takie same: użyj nn.Identity() (nic nie zmienia)
        self.resizer =

        # 3) Zapisz skalę sqrt(d_model) (jako liczbę), aby nie liczyć jej w każdej iteracji forward()
        #    Użyj: math.sqrt(d_model)
        self.embed_scale =

        # 4) Utwórz moduł pozycyjny i dropout
        #    Użyj: PositionalSignal(d_model), nn.Dropout(...)
        self.pos_module =
        self.regularization =

    def forward(self, tokens):
        # tokens powinno mieć dtype=torch.long, bo nn.Embedding przyjmuje indeksy całkowite
        if not tokens.dtype == torch.long:
            raise TypeError(f"Oczekiwano torch.long, otrzymano {tokens.dtype}")

        # TODO (student)
        # 1) Zamień tokeny na embeddingi:
        #    [batch, seq_len] -> [batch, seq_len, d_embed]
        #    Użyj: self.word_lookup(tokens)
        #
        # 2) Przeskaluj embeddingi przez sqrt(d_model)
        #    (mnożenie przez self.embed_scale)
        x =

        # 3) Dopasuj wymiar do d_model (jeśli trzeba)
        #    [batch, seq_len, d_embed] -> [batch, seq_len, d_model]
        x =

        # 4) Dodaj kodowanie pozycyjne (ten sam kształt co x)
        #    Użyj: self.pos_module(x)
        x =

        # 5) Zastosuj dropout i zwróć wynik
        #    To jest gotowe wejście do kolejnych bloków transformera (np. enkodera)
        return

### Zadanie 2: Mechanizm Wielogłowej Uwagi (Multi-Head Attention)

W tym etapie zaimplementujecie serce modelu Transformer. Mechanizm ten pozwala sieci skupić się na różnych fragmentach zdania jednocześnie, co opisano w sekcjach 3.2.1 oraz 3.2.2 artykułu. Zamiast liczyć jedną "uśrednioną" uwagę, model dzieli dane na wiele "głów", z których każda może uczyć się innych zależności językowych.

Kluczowe zasady mechanizmu:

* **Skalowany Iloczyn Skalarny (Scaled Dot-Product)** - Podstawą jest wzór $Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$. Dzielenie przez $\sqrt{d_k}$ zapobiega powstawaniu zbyt dużych wartości, które mogłyby "zamrozić" proces uczenia się (małe gradienty w funkcji softmax).

* **Wielogłowość (Multi-Head)** - Zamiast jednej operacji na dużym wektorze, rzutujemy zapytania (Q), klucze (K) i wartości (V) $h$ razy do mniejszych wymiarów. W pracy przyjęto $h=8$ głowic, z których każda operuje na wymiarze $d_k = 64$.
* **Maskowanie** - W dekoderze musimy zablokować możliwość "zaglądania w przyszłość". Robimy to, dodając do wyników przed softmaxem bardzo małą liczbę ($-\infty$), co po nałożeniu funkcji softmax wyzeruje te połączenia.

* **Uniwersalność** - Mechanizm musi obsługiwać zarówno samo-uwagę (self-attention), jak i uwagę krzyżową (cross-attention), gdzie Q pochodzi z dekodera, a K i V z enkodera.

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadAttentionModule(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        # Liczba głowic musi dzielić wymiar modelu bez reszty
        assert d_model % num_heads == 0, "d_model musi dzielić się przez num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Wymiar jednej głowy

        # TODO (student)
        # 1) Utwórz 3 warstwy liniowe do projekcji:
        #    - query (Q)
        #    - key   (K)
        #    - value (V)
        #    Każda mapuje z d_model -> d_model
        #    Użyj: nn.Linear(d_model, d_model)
        self.q_linear =
        self.k_linear =
        self.v_linear =

        # 2) Utwórz projekcję końcową po scaleniu wszystkich głowic
        #    Również: d_model -> d_model
        self.out_projection =

        # 3) Dropout na wagach uwagi (po softmax)
        self.attention_dropout =

    def split_heads(self, x, batch_size):
        """
        Rozdziela reprezentację na wiele głów.

        Wejście:
            x: [batch, seq_len, d_model]
        Wyjście:
            [batch, num_heads, seq_len, d_k]
        """
        # Krok 1: rozbij ostatni wymiar d_model na [num_heads, d_k]
        #         użyj .view(batch_size, seq_len, num_heads, d_k)
        # Krok 2: zamień kolejność wymiarów, aby num_heads było przed seq_len
        #         użyj .transpose(1, 2)
        return

    def forward(self, query, key, value, mask=None):
        # query: [batch, seq_len_q, d_model]
        # key:   [batch, seq_len_k, d_model]
        # value: [batch, seq_len_k, d_model]
        # Uwaga: seq_len_q i seq_len_k mogą być różne (cross-attention)
        batch_size = query.size(0)

        # TODO (student)
        # 1) Przepuść query/key/value przez odpowiednie warstwy liniowe
        #    i rozdziel na głowy funkcją split_heads(...)
        #
        # Po tym kroku:
        # q, k, v -> [batch, num_heads, seq_len, d_k]
        q =
        k =
        v =

        # 2) Oblicz "scores" = QK^T / sqrt(d_k)
        #    Użyj:
        #    - k.transpose(-2, -1), aby zamienić [seq_len_k, d_k] -> [d_k, seq_len_k]
        #    - torch.matmul(q, ...)
        #    - podziel przez math.sqrt(self.d_k)
        #
        # Wynik:
        # scores -> [batch, num_heads, seq_len_q, seq_len_k]
        scores =

        # 3) Jeśli jest maska, zablokuj wybrane pozycje przed softmax
        #    Użyj: masked_fill(mask == 0, -1e9)
        #    Dzięki temu softmax da ~0 na zablokowanych pozycjach.
        #
        # WAŻNE: maska musi dać się rozgłosić (broadcast) do kształtu scores.
        # Typowe kształty maski:
        # - [batch, 1, 1, seq_len_k]      (padding mask)
        # - [batch, 1, seq_len_q, seq_len_k]


        # 4) Zamień scores na wagi uwagi:
        #    - softmax po ostatnim wymiarze (po kluczach)
        #    - attention_dropout na wagach
        #    Użyj: torch.softmax(scores, dim=-1)
        weights =
        weights =

        # 5) Policz ważoną sumę wartości V:
        #    context = weights @ v
        #
        # Kształt po matmul:
        # context -> [batch, num_heads, seq_len_q, d_k]
        context =

        # 6) Scal głowy z powrotem do jednego wektora:
        #    a) transpose(1, 2): [batch, seq_len_q, num_heads, d_k]
        #    b) contiguous(): ważne przed view po transpose
        #    c) view(..., d_model): bo num_heads * d_k = d_model
        #
        # Wynik:
        # context -> [batch, seq_len_q, d_model]
        context =

        # 7) Finalna projekcja wyjściowa
        #    Użyj: self.out_projection(context)
        return

### Zadanie 3: Pozycyjna Sieć Feed-Forward
Zgodnie z opisem w sekcji 3.3 oryginału, sieć ta jest stosowana do każdej pozycji w sekwencji całkowicie niezależnie i identycznie. Oznacza to, że każde słowo (lub token) przechodzi przez dokładnie te same operacje liniowe, co pozwala na łatwą równoległość obliczeń.

Kluczowe parametry techniczne:
- **Architektura** - Składa się z dwóch przekształceń liniowych, pomiędzy którymi znajduje się funkcja aktywacji ReLU.
- **Wymiarowość** - Warstwa wejściowa i wyjściowa mają rozmiar $d_{model} = 512$, natomiast wewnętrzna warstwa ukryta jest znacznie szersza i wynosi $d_{ff} = 2048$.
- **Równanie** - Proces ten można zapisać jako $FFN(x) = max(0, xW_1 + b_1)W_2 + b_2$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionWiseFFN(nn.Module):
    """
    Dwuwarstwowa sieć Feed-Forward stosowana niezależnie na każdej pozycji sekwencji.

    Działa na ostatnim wymiarze tensora:
    [batch, seq_len, d_model] -> [batch, seq_len, d_ff] -> [batch, seq_len, d_model]
    """
    def __init__(self, d_model, d_ff, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        # TODO (student) — inicjalizacja warstw
        # 1) Pierwsza warstwa liniowa rozszerza wymiar reprezentacji:
        #    d_model -> d_ff
        #    Użyj: nn.Linear(d_model, d_ff)
        self.w_expand =

        # 2) Druga warstwa liniowa zwęża reprezentację z powrotem:
        #    d_ff -> d_model
        #    Użyj: nn.Linear(d_ff, d_model)
        self.w_shrink =

        # 3) (Opcjonalnie) zainicjalizuj wagi, np. metodą Xavier uniform,
        #    aby trening był stabilniejszy na starcie.
        #    Użyj: nn.init.xavier_uniform_(...)


        # 4) Dropout po aktywacji (na warstwie ukrytej)
        #    Użyj: nn.Dropout(dropout_rate)
        self.dropout =

    def forward(self, x):
        # x powinien mieć kształt [batch, seq_len, d_model]
        # Sprawdzamy tylko ostatni wymiar, bo FFN działa "po cechach"
        if x.size(-1) != self.d_model:
            raise ValueError(
                f"Błąd: oczekiwano ostatniego wymiaru = d_model ({self.d_model}), "
                f"otrzymano {x.size(-1)}"
            )

        # TODO (student) — przepływ danych
        # Krok 1) Rozszerzenie wymiaru + aktywacja nieliniowa
        #         Użyj: self.w_expand(x), a potem F.relu(...)
        # Wynik: [batch, seq_len, d_ff]
        hidden_state =

        # Krok 2) Dropout na reprezentacji ukrytej
        #         Użyj: self.dropout(...)
        hidden_state =

        # Krok 3) Projekcja z powrotem do d_model
        #         Użyj: self.w_shrink(...)
        # Wynik: [batch, seq_len, d_model]
        output =

        return output

In [ ]:
net = PositionWiseFFN(d_model = 512,  d_ff =2048)
print(net)

PositionWiseFFN(
  (w_expand): Linear(in_features=512, out_features=2048, bias=True)
  (w_shrink): Linear(in_features=2048, out_features=512, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


### Zadanie 4: Kompletna Warstwa Enkodera

Warstwa enkodera składa się z dwóch głównych podwarstw: mechanizmu Multi-Head Self-Attention oraz sieci Position-wise Feed-Forward. Kluczowym aspektem tej konstrukcji jest to, że każda z tych podwarstw jest otoczona połączeniem rezydualnym, po którym następuje normalizacja warstwowa (Layer Normalization).

Zasady konstrukcji (Sekcja 3.1):
- **Struktura podwarstwy** - Wyjście każdej podwarstwy definiuje równanie $\text{LayerNorm}(x + \text{Sublayer}(x))$, gdzie $\text{Sublayer}(x)$ to funkcja realizowana przez dany moduł (uwaga lub FFN).
- **Regularyzacja** - Zgodnie z Sekcją 5.4, dropout należy zaaplikować do wyjścia każdej podwarstwy, zanim zostanie ono dodane do wejścia podwarstwy i znormalizowane.
- **Spójność wymiarów** - Wszystkie podwarstwy w modelu, a także warstwy osadzeń, generują wyjścia o wymiarze $d_{model} = 512$, co umożliwia działanie połączeń rezydualnych.

In [ ]:
import torch
import torch.nn as nn

class EncoderBlock(nn.Module):
    """
    Pojedyncza warstwa kodera (Sekcja 3.1).
    Łączy mechanizm Self-Attention, FFN oraz Residual Connections.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # TODO Inicjalizacja modułów
        # 1. Główny mechanizm samo-uwagi (Self-Attention)
        self.self_attention_block =

        # 2. Sieć Feed-Forward (FFN)
        self.feed_forward_block =

        # 3. Warstwy normalizacji dla obu podwarstw (LayerNorm)
        self.norm_attention =
        self.norm_ffn =

        # 4. Dropout stosowany przed dodaniem do połączenia rezydualnego
        self.dropout_layer =

    def forward(self, x, padding_mask=None):
        """
        Realizacja struktury: LayerNorm(x + Dropout(Sublayer(x)))
        """

        # TODO Pierwsza podwarstwa - Attention
        # Krok 1: Obliczenie samo-uwagi (Query, Key i Value to ten sam tensor x)
        attn_out =

        # Krok 2: Dropout i połączenie rezydualne z normalizacją
        x = self.norm_attention(x + self.dropout_layer(attn_out))

        # TODO Druga podwarstwa - FFN
        # Krok 3: Przetworzenie przez sieć Feed-Forward
        ffn_out =

        # Krok 4: Dropout i drugie połączenie rezydualne z normalizacją
        x = self.norm_ffn(x + self.dropout_layer(ffn_out))

        return x

In [ ]:
net = EncoderBlock( d_model = 512, d_ff =2048, num_heads=8, dropout=0.1)
print(net)

EncoderBlock(
  (self_attention_block): MultiHeadAttentionModule(
    (q_linear): Linear(in_features=512, out_features=512, bias=True)
    (k_linear): Linear(in_features=512, out_features=512, bias=True)
    (v_linear): Linear(in_features=512, out_features=512, bias=True)
    (out_projection): Linear(in_features=512, out_features=512, bias=True)
    (attention_dropout): Dropout(p=0.1, inplace=False)
  )
  (feed_forward_block): PositionWiseFFN(
    (w_expand): Linear(in_features=512, out_features=2048, bias=True)
    (w_shrink): Linear(in_features=2048, out_features=512, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm_attention): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (norm_ffn): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout_layer): Dropout(p=0.1, inplace=False)
)


### Zadanie 5: Warstwa Dekodera i Mechanizm Maskowania

Zgodnie z Sekcją 3.1, dekoder składa się z trzech podwarstw, z których każda otoczona jest połączeniem rezydualnym i normalizacją warstwową:

- **Masked Self-Attention** - Zapobiega "zaglądaniu w przyszłość" podczas generowania tekstu.
- **Cross-Attention** - Pozwala dekoderowi skupić się na odpowiednich fragmentach zdania wejściowego (pochodzącego z enkodera).
- **Feed-Forward Network (FFN)** - Przetwarza cechy dla każdej pozycji.

Kluczowe szczegóły implementacyjne:
- **Causal Masking (Maska przyczynowa)** - Wykorzystujemy macierz trójkątną górną, aby wypełnić niedozwolone pozycje wartością $-\infty$ przed funkcją softmax. Dzięki temu przewidywanie dla pozycji $i$ zależy tylko od znanych wyjść na pozycjach mniejszych niż $i$.
- **Regularyzacja** - Podobnie jak w enkoderze, dropout stosujemy do wyjścia każdej podwarstwy przed dodaniem go do wejścia i normalizacją.

In [ ]:
import torch
import torch.nn as nn

class EncoderBlock(nn.Module):
    """
    Pojedynczy blok enkodera:
    1) Self-Attention
    2) Feed-Forward Network (FFN)

    Każda podwarstwa jest opakowana w:
    residual connection + dropout + LayerNorm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # TODO (student) — inicjalizacja modułów
        # 1) Mechanizm self-attention
        #    Użyj wcześniej zdefiniowanego modułu MultiHeadAttentionModule
        self.self_attention_block =

        # 2) Sieć Feed-Forward działająca na każdej pozycji niezależnie
        #    Użyj modułu PositionWiseFFN(d_model, d_ff, dropout)
        self.feed_forward_block =

        # 3) Dwie osobne warstwy LayerNorm:
        #    - po attention
        #    - po FFN
        #    Użyj: nn.LayerNorm(d_model)
        self.norm_attention =
        self.norm_ffn =

        # 4) Dropout stosowany na wyjściu każdej podwarstwy
        #    przed dodaniem połączenia rezydualnego
        self.dropout_layer =

    def forward(self, x, padding_mask=None):
        """
        Wejście:
            x: [batch, seq_len, d_model]
            padding_mask: maska paddingu (opcjonalna), broadcastowalna do attention scores

        Wyjście:
            x: [batch, seq_len, d_model]
        """

        # TODO (student) — podwarstwa 1: Self-Attention
        # Krok 1) Self-attention:
        # query = key = value = x (bo to self-attention)
        # Wynik ma ten sam kształt co x: [batch, seq_len, d_model]
        attn_out =

        # Krok 2) Dropout + residual connection + LayerNorm
        # Schemat:
        # x <- LayerNorm( x + Dropout(attn_out) )
        x =

        # TODO (student) — podwarstwa 2: Feed-Forward
        # Krok 3) Przepuść wynik przez FFN
        # Kształt pozostaje: [batch, seq_len, d_model]
        ffn_out =

        # Krok 4) Dropout + residual connection + LayerNorm
        # Schemat:
        # x <- LayerNorm( x + Dropout(ffn_out) )
        x =

        return x

In [ ]:
net = TransformerDecoderLayer( d_model = 512, d_ff =2048, num_heads=8, dropout=0.1)
print(net)

TransformerDecoderLayer(
  (self_attn): MultiHeadAttentionModule(
    (q_linear): Linear(in_features=512, out_features=512, bias=True)
    (k_linear): Linear(in_features=512, out_features=512, bias=True)
    (v_linear): Linear(in_features=512, out_features=512, bias=True)
    (out_projection): Linear(in_features=512, out_features=512, bias=True)
    (attention_dropout): Dropout(p=0.1, inplace=False)
  )
  (cross_attn): MultiHeadAttentionModule(
    (q_linear): Linear(in_features=512, out_features=512, bias=True)
    (k_linear): Linear(in_features=512, out_features=512, bias=True)
    (v_linear): Linear(in_features=512, out_features=512, bias=True)
    (out_projection): Linear(in_features=512, out_features=512, bias=True)
    (attention_dropout): Dropout(p=0.1, inplace=False)
  )
  (ffn_block): PositionWiseFFN(
    (w_expand): Linear(in_features=512, out_features=2048, bias=True)
    (w_shrink): Linear(in_features=2048, out_features=512, bias=True)
    (dropout): Dropout(p=0.1, inplace=

### Zadanie 6: Budowa Stosów Enkodera i Dekodera

W tym zadaniu zaimplementujecie finalne struktury zarządzające wieloma warstwami obliczeniowymi. Kluczowym wyzwaniem jest tutaj poprawne przekazywanie danych: wynik pracy ostatniej warstwy enkodera musi trafić do każdej warstwy dekodera jako źródło informacji dla mechanizmu cross-attention.

Kluczowe założenia (Sekcja 3.1):
- **Liczba warstw ($N$)** - Zarówno enkoder, jak i dekoder składają się ze stosu $N=6$ identycznych warstw.+1
- **Przepływ danyc** Każda podwarstwa w całym stosie produkuje wyjście o wymiarze $d_{model} = 512$, co pozwala na zachowanie ciągłości połączeń rezydualnych wewnątrz całego stosu.
- **Mechanizm Attention** W warstwach dekodera zapytania (Queries) pochodzą z poprzedniej warstwy dekodera, natomiast klucze (Keys) i wartości (Values) pochodzą bezpośrednio z końcowego wyjścia stosu enkodera.

In [ ]:
import copy
import torch
import torch.nn as nn

class TransformerStack(nn.Module):
    """
    Uniwersalny stos N warstw (np. enkodera albo dekodera).

    Idea:
    - dostajemy pojedynczy blok (warstwę),
    - tworzymy N niezależnych kopii,
    - przepuszczamy dane kolejno przez każdą warstwę,
    - na końcu opcjonalnie stosujemy LayerNorm.
    """
    def __init__(self, layer_block, num_layers):
        super().__init__()

        # TODO (student) — przechowywanie warstw
        # 1) Użyj nn.ModuleList(...), aby PyTorch poprawnie rejestrował warstwy
        #    (dzięki temu ich parametry będą widoczne dla optymalizatora i .to(device)).
        #
        # 2) WAŻNE: każda warstwa powinna być osobną kopią (osobne wagi),
        #    więc nie używaj samego: [layer_block for _ in range(num_layers)]
        #    bo to doda TEN SAM obiekt wiele razy.
        #
        #    Użyj: copy.deepcopy(layer_block)
        self.layers =

        # 3) (Opcjonalnie) końcowa normalizacja po całym stosie.
        #    Zakładamy, że blok ma atrybut d_model.
        #    Użyj: nn.LayerNorm(layer_block.d_model)
        self.final_norm =

    def forward(self, x, *args, **kwargs):
        """
        Sekwencyjne przetwarzanie:
        x -> warstwa1 -> warstwa2 -> ... -> warstwaN -> final_norm

        *args i **kwargs pozwalają przekazywać dodatkowe argumenty
        (np. maski, encoder output itp.) bez przepisywania tej klasy
        osobno dla enkodera i dekodera.
        """
        # TODO (student)
        # Przejdź po wszystkich warstwach w self.layers i za każdym razem:
        # x = layer(x, *args, **kwargs)


        # Zastosuj końcową normalizację i zwróć wynik
        return


class FullTransformerEncoder(nn.Module):
    """
    Pełny enkoder:
    1) embedding + pozycje
    2) stos N bloków enkodera
    """
    def __init__(self, vocab_size, d_model, d_ff, num_heads, num_layers, dropout=0.1):
        super().__init__()

        # TODO (student)
        # 1) Moduł wejściowy (embedding + pozycje + dropout)
        #    Użyj wcześniej przygotowanego TransformerInput
        self.input_module =

        # 2) Zbuduj pojedynczy blok enkodera
        encoder_layer =

        # 3) Utwórz stos N warstw enkodera (z niezależnymi kopiami wag)
        #    Użyj klasy TransformerStack
        self.stack =

    def forward(self, x, mask=None):
        """
        Wejście:
            x: [batch, src_seq_len] (tokeny)
            mask: maska paddingu dla enkodera (opcjonalna)

        Wyjście:
            [batch, src_seq_len, d_model]
        """
        # Krok 1) Zamień tokeny na reprezentacje wektorowe + dodaj pozycje
        x = self.input_module(x)

        # Krok 2) Przepuść przez stos bloków enkodera
        # Uwaga: EncoderBlock przyjmuje argument padding_mask, więc przekazujemy go nazwą
        x = self.stack(x, padding_mask=mask)

        return x


class FullTransformerDecoder(nn.Module):
    """
    Pełny dekoder:
    1) embedding + pozycje
    2) stos N bloków dekodera
    """
    def __init__(self, vocab_size, d_model, d_ff, num_heads, num_layers, dropout=0.1):
        super().__init__()

        # TODO (student)
        # 1) Moduł wejściowy dla tokenów docelowych (tgt)
        self.input_module =

        # 2) Utwórz pojedynczy blok dekodera
        #    (zakładamy, że masz klasę TransformerDecoderLayer)
        decoder_layer =

        # 3) Zbuduj stos N bloków dekodera
        self.stack =

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        """
        Wejście:
            x: [batch, tgt_seq_len]              - tokeny wejściowe dekodera
            enc_output: [batch, src_seq_len, d_model] - wyjście enkodera
            src_mask: maska dla uwagi krzyżowej (encoder-decoder attention)
            tgt_mask: maska dla self-attention dekodera (padding +/lub causal mask)

        Wyjście:
            [batch, tgt_seq_len, d_model]
        """
        # Krok 1) Embedding + pozycje dla sekwencji docelowej
        x = self.input_module(x)

        # Krok 2) Przepuszczenie przez wszystkie warstwy dekodera
        # Każda warstwa dostaje:
        # - aktualny stan dekodera x
        # - wyjście enkodera enc_output
        # - maski src/tgt
        #
        # Zakładamy, że TransformerDecoderLayer ma forward zgodny z:
        # layer(x, enc_output, src_mask=None, tgt_mask=None)
        x = self.stack(x, enc_output, src_mask=src_mask, tgt_mask=tgt_mask)

        return x

### Zadanie 7: Pełna Architektura Transformer

W tym zadaniu zbudujecie finalną klasę, która zarządza przepływem danych między koderem a dekoderem. Zgodnie z **Sekcją 3** publikacji, model ten działa w sposób auto-regresywny generuje jeden symbol na raz, wykorzystując poprzednio stworzone znaki jako dodatkowe wejście.

**Kluczowe aspekty integracji (Sekcje 3.1 - 3.4):**

* **Współdzielenie wag -** Często stosowaną praktyką jest używanie tej samej macierzy wag dla osadzeń wejściowych, wyjściowych oraz warstwy liniowej przed funkcją softmax.

* **Skalowanie -** Pamiętajcie o pomnożeniu wag w warstwach osadzeń przez .

* **Przesunięcie wyjścia -** Podczas treningu (teacher forcing), wejście dekodera jest przesunięte o jedną pozycję w prawo względem celu, co zapewnia, że przewidywanie dla pozycji  opiera się tylko na pozycjach mniejszych niż .

* **Stabilność numeryczna -** Zamiast czystej funkcji softmax, na wyjściu stosuje się często `log_softmax`, co poprawia stabilność obliczeń podczas wyliczania funkcji straty.


In [ ]:
import torch
import torch.nn as nn

class TransformerModel(nn.Module):
    """
    Kompletna architektura Transformer:
    - encoder (przetwarza sekwencję źródłową)
    - decoder (generuje reprezentacje sekwencji docelowej)
    - warstwa wyjściowa (projekcja do słownika)
    """
    def __init__(self, encoder_stack, decoder_stack, target_vocab_size):
        super().__init__()
        self.encoder = encoder_stack
        self.decoder = decoder_stack

        # TODO (student) — głowica wyjściowa
        # 1) Warstwa liniowa mapująca z d_model -> target_vocab_size
        #    Dzięki temu dla każdej pozycji sekwencji dostajemy logity dla wszystkich tokenów.
        #    Użyj: nn.Linear(..., target_vocab_size)
        #
        # Uwaga: decoder_stack musi mieć atrybut d_model (np. self.d_model = d_model w klasie dekodera)
        self.final_projection =

        # 2) LogSoftmax po ostatnim wymiarze (wymiar słownika)
        #    Zwracamy log-prawdopodobieństwa.
        #    Użyj: nn.LogSoftmax(dim=-1)
        #
        # Uwaga praktyczna:
        # - jeśli używasz nn.NLLLoss -> log_softmax jest OK
        # - jeśli używasz nn.CrossEntropyLoss -> zwykle zwraca się surowe logity (bez softmax/log_softmax)
        self.log_softmax =

    def create_start_token_shift(self, target_tokens):
        """
        Przygotowuje wejście dekodera (teacher forcing):
        przesuwa target o 1 w prawo i wstawia token startowy na początek.

        Przykład:
        target        = [t1, t2, t3, t4]
        shifted_input = [<SOS>, t1, t2, t3]
        """
        # TODO (student)
        # 1) Pobierz rozmiar batcha
        #    Użyj: target_tokens.size(0)
        batch_size =

        # 2) Utwórz kolumnę tokenów startowych o kształcie [batch_size, 1]
        #    Użyj: torch.zeros(...), jeśli indeks startowego tokenu = 0
        #    Ważne: ustaw dtype i device takie same jak w target_tokens
        start_tokens =

        # 3) Sklej token startowy z targetem bez ostatniego elementu
        #    target_tokens[:, :-1] usuwa ostatni token z każdej sekwencji
        #    Użyj: torch.cat([...], dim=1)
        #
        # Wynik ma ten sam kształt co target_tokens: [batch_size, tgt_seq_len]
        return

    def forward(self, source, target, src_mask=None, tgt_mask=None):
        """
        Główny przepływ danych przez model.

        Wejście:
            source: [batch_size, src_seq_len]  - tokeny źródłowe
            target: [batch_size, tgt_seq_len]  - tokeny docelowe (prawdziwe, do teacher forcing)
            src_mask: maska dla enkodera / cross-attention (opcjonalna)
            tgt_mask: maska dla self-attention dekodera (np. causal mask + padding, opcjonalna)

        Wyjście:
            log_probs: [batch_size, tgt_seq_len, target_vocab_size]
        """
        # 1) Przygotowanie wejścia dekodera: przesunięcie targetu w prawo
        shifted_target = self.create_start_token_shift(target)

        # TODO (student) — przepływ przez encoder i decoder

        # 2) Encoder: kodowanie sekwencji źródłowej
        #    Wynik to "pamięć" dla dekodera (encoder memory)
        #    Kształt: [batch_size, src_seq_len, d_model]
        encoded_memory =

        # 3) Decoder:
        #    - wejście: shifted_target
        #    - pamięć z enkodera: encoded_memory
        #    - maski: src_mask, tgt_mask
        #
        #    Wynik: [batch_size, tgt_seq_len, d_model]
        decoded_sequence =

        # 4) Projekcja do rozmiaru słownika (logity dla każdego tokenu)
        #    Kształt: [batch_size, tgt_seq_len, target_vocab_size]
        logits =

        # 5) Zamiana logitów na log-prawdopodobieństwa
        #    (jeśli trenujesz z NLLLoss)
        return

## Cz. II
1. Przygotowanie danych.
2. Strategia uczenia.
3. Mechanizm wnioskowania.
4. Ewaluacja i wizualizacja.